In [ ]:
import os
import pandas as pd

# 1. Load stock code mapping table (stock name -> stock code)
mapping = pd.read_csv("Eastmoney_report_pdf_download\\HS300.csv", dtype={"股票代码": str})
mapping["股票代码"] = mapping["股票代码"].str.zfill(6)
name2code = dict(zip(mapping["股票简称"], mapping["股票代码"]))

# 2. Define rating keyword mapping (0–4 five-level ratings)
rating_dict = {
    "卖出": 0,          # Sell
    "减持": 0,          # Reduce
    "回避": 0,          # Avoid
    "中性": 1,          # Neutral
    "观望": 1,          # Observe
    "持有": 1,          # Hold
    "买入": 2,          # Buy
    "增持": 3,          # Increase
    "推荐": 3,          # Recommend
    "强烈推荐": 4       # Strong Recommend
}

# Define log file path (in current directory)
log_file = os.path.join(os.getcwd(), "report_label_errors.txt")

# Initialize log file
with open(log_file, "w", encoding="utf-8") as logf:
    logf.write("Error log started\n")

def extract_rating_from_file(filepath, max_lines=5):
    """Read only the first few lines of a file to extract rating label"""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            lines = []
            for _ in range(max_lines):
                try:
                    lines.append(next(f).strip())
                except StopIteration:
                    break
        text_to_check = " ".join(lines)
        for k, v in rating_dict.items():
            if k in text_to_check:
                return v
        # No match found → -1, write to log
        with open(log_file, "a", encoding="utf-8") as logf:
            logf.write(f"No rating keyword found: {filepath}\n")
        return -1
    except Exception as e:
        # Error reading file → -1, write to log
        with open(log_file, "a", encoding="utf-8") as logf:
            logf.write(f"Failed to read {filepath}: {e}\n")
        return -1


# 3. Traverse report directories (Chinese original version)
data = []
base_dir_cn = "reports_txt_by_quarter"              # Chinese original directory
base_dir_en = "reports_txt_by_quarter_cleaned_en"  # English cleaned version directory

for quarter in sorted(os.listdir(base_dir_cn)):  # Sort by time sequence
    quarter_path_cn = os.path.join(base_dir_cn, quarter)
    quarter_path_en = os.path.join(base_dir_en, quarter)
    if not os.path.isdir(quarter_path_cn) or not os.path.isdir(quarter_path_en):
        continue
    
    quarter_key = quarter.replace("_", "")
    
    for company_name in os.listdir(quarter_path_cn):
        company_path_cn = os.path.join(quarter_path_cn, company_name)
        company_path_en = os.path.join(base_dir_en, quarter, company_name)
        if not os.path.isdir(company_path_cn) or not os.path.isdir(company_path_en):
            continue
        
        if company_name not in name2code:
            continue
        stock_code = name2code[company_name]
        
        # Traverse Chinese reports to extract rating
        for root, _, files in os.walk(company_path_cn):
            for file in files:
                if file.endswith(".txt"):
                    file_path_cn = os.path.join(root, file)
                    label = extract_rating_from_file(file_path_cn, max_lines=5)
                    
                    # Find corresponding English report
                    file_path_en = os.path.join(company_path_en, file)
                    if not os.path.exists(file_path_en):
                        continue
                    
                    try:
                        with open(file_path_en, "r", encoding="utf-8") as f:
                            text_en = f.read().strip()
                        if len(text_en) > 50:
                            data.append({
                                "quarter": quarter_key,
                                "company": company_name,
                                "stock_code": stock_code,
                                "text_en": text_en,
                                "label": label
                            })
                    except Exception as e:
                        with open(log_file, "a", encoding="utf-8") as logf:
                            logf.write(f"Skip file {file_path_en}: {e}\n")

# 4. Convert to DataFrame and sort by quarter
df = pd.DataFrame(data)
df = df.sort_values("quarter")

# 5. Time-series split: first 80% quarters for training, last 20% for testing
unique_quarters = df["quarter"].unique()
split_point = int(len(unique_quarters) * 0.8)

train_quarters = unique_quarters[:split_point]
test_quarters = unique_quarters[split_point:]

train_df = df[df["quarter"].isin(train_quarters)]
test_df = df[df["quarter"].isin(test_quarters)]

train_df.to_csv("train_timeseries_reportlabel.csv", index=False)
test_df.to_csv("test_timeseries_reportlabel.csv", index=False)

print(f"Dataset generation complete: train set {train_quarters[0]}–{train_quarters[-1]} → test set {test_quarters[0]}–{test_quarters[-1]}")
print(f"Error log written to {log_file}")

生成完成: 训练集 2017Q1–2023Q4 → 测试集 2024Q1–2025Q4
错误日志已写入 d:\MFIN\7036\report_label_errors.txt
